# CatBoost 建模 v2.1（日期化时间差 + 稳健参数 + 多种子可选）

In [5]:
import os, json, numpy as np, pandas as pd
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold
from catboost import CatBoostClassifier, Pool
SEED=2025
np.random.seed(SEED)
SECONDS_PER_DAY=86400.0
NON_FEATURE_COLS=['id','label']

def to_dt(series):
    return pd.to_datetime(pd.to_numeric(series, errors='coerce'), unit='s', utc=True)


In [6]:
TRAIN_CSV = 'train/train.csv'
TRAIN_STMT_FEAT = 'train/train_statement_feature_v2.csv'
TEST_CSV = 'testaa/testaa.csv'                      # 如果没有可置为 None
TEST_STMT_FEAT = 'testaa/testaa_statement_feature_v2.csv'  # 如果没有可置为 None
OUT_SUBMISSION = 'output_Cat/submission.csv'
SAVE_OOF = 'output_Cat/oof_pred.csv'
SAVE_INFO = 'output_Cat/cv_info.json'
DROP_HIGH_DRIFT_COLS=False
HIGH_DRIFT_COLS=['zip_code']
ENSEMBLE_SEEDS=[2025]  # 可改为 [2025,2026,2027]

In [7]:
def _coerce_numeric(series):
    return pd.to_numeric(series, errors='coerce')

def _safe_div(a,b):
    try:
        b=b.replace(0,np.nan)
    except Exception:
        b=np.where(b==0,np.nan,b)
    return a/b

def base_feature_engineering(df: pd.DataFrame, is_train=True):
    X=df.copy()
    for col in X.columns:
        if X[col].dtype=='O':
            X[col]=X[col].replace(['',' ','nan','NaN','NULL','None'], np.nan)
    if 'level' in X.columns:
        X['level']=X['level'].astype(str)
        X['grade']=X['level'].str[0]
        X['subgrade']=pd.to_numeric(X['level'].str[1:], errors='coerce')
        X['grade_rank']=X['grade'].map({'A':5,'B':4,'C':3,'D':2,'E':1}).fillna(0).astype('Int64')
    if {'record_time','issue_time'}.issubset(X.columns):
        rec=to_dt(X['record_time']); iss=to_dt(X['issue_time'])
        X['days_issue_to_record']=((rec-iss).dt.total_seconds()/SECONDS_PER_DAY).clip(lower=0)
    if {'record_time','history_time'}.issubset(X.columns):
        rec=to_dt(X['record_time']); his=to_dt(X['history_time'])
        X['days_history_to_record']=((rec-his).dt.total_seconds()/SECONDS_PER_DAY).clip(lower=0)
    if {'balance','balance_limit'}.issubset(X.columns):
        X['balance_utilization']=_safe_div(_coerce_numeric(X['balance']), _coerce_numeric(X['balance_limit']))
        X['balance_utilization_sqrt']=np.sqrt(X['balance_utilization'].clip(lower=0))
    if {'balance_accounts','total_accounts'}.issubset(X.columns):
        X['acct_utilization']=_safe_div(_coerce_numeric(X['balance_accounts']), _coerce_numeric(X['total_accounts']))
    if {'loan','balance_limit'}.issubset(X.columns):
        X['loan_to_limit']=_safe_div(_coerce_numeric(X['loan']), _coerce_numeric(X['balance_limit']))
    if {'loan','term'}.issubset(X.columns):
        X['loan_per_month']=_safe_div(_coerce_numeric(X['loan']), _coerce_numeric(X['term']))
    for c in ['title','career','zip_code','residence','term','syndicated','installment','level','grade']:
        if c in X.columns:
            if c in ['level','grade']:
                X[c]=X[c].astype('object')
            else:
                X[c]=pd.to_numeric(X[c], errors='coerce').astype('Int64')
    for col in ['issue_time','record_time','history_time']:
        if col in X.columns:
            X.drop(columns=[col], inplace=True)
    return X

def prepare_dataset(train_csv, stmt_feat_csv, test_csv=None, test_stmt_feat_csv=None):
    train=pd.read_csv(train_csv)
    stmt=pd.read_csv(stmt_feat_csv)
    df=train.merge(stmt, on='id', how='left', suffixes=('', '_stmt'))
    df['stmt_missing']=df['has_statement'].apply(lambda x: 0 if x==1 else 1) if 'has_statement' in df.columns else 1
    y=df['label'].astype(int)
    X=df.drop(columns=['label'])
    if DROP_HIGH_DRIFT_COLS:
        for c in HIGH_DRIFT_COLS:
            if c in X.columns: X.drop(columns=[c], inplace=True)
    X_test=None
    if test_csv is not None and os.path.exists(test_csv):
        test=pd.read_csv(test_csv)
        if test_stmt_feat_csv is not None and os.path.exists(test_stmt_feat_csv):
            stmt_t=pd.read_csv(test_stmt_feat_csv)
            X_test=test.merge(stmt_t, on='id', how='left', suffixes=('', '_stmt'))
            X_test['stmt_missing']=X_test['has_statement'].apply(lambda x: 0 if x==1 else 1) if 'has_statement' in X_test.columns else 1
        else:
            X_test=test.copy(); X_test['stmt_missing']=1
        if DROP_HIGH_DRIFT_COLS:
            for c in HIGH_DRIFT_COLS:
                if c in X_test.columns: X_test.drop(columns=[c], inplace=True)
    return X,y,X_test

def get_cat_cols(df: pd.DataFrame):
    cat_cols=[]
    for c in ['title','career','zip_code','residence','term','syndicated','installment','level','grade']:
        if c in df.columns: cat_cols.append(c)
    for c in df.select_dtypes(include='object').columns:
        if c not in cat_cols: cat_cols.append(c)
    cat_cols=[c for c in cat_cols if c not in NON_FEATURE_COLS and c in df.columns]
    return sorted(list(set(cat_cols)))

from pandas.api.types import is_numeric_dtype

def fill_missing(df: pd.DataFrame, cat_cols):
    X = df.copy()
    for c in X.columns:
        if c in NON_FEATURE_COLS:
            continue
        if c in cat_cols:
            X[c] = X[c].astype('object').fillna('Unknown')
            continue
        if is_numeric_dtype(X[c]):
            X[c] = X[c].fillna(X[c].median())
            continue
        xnum = pd.to_numeric(X[c], errors='coerce')
        if xnum.notna().any():
            X[c] = xnum.fillna(xnum.median())
        else:
            X[c] = X[c].astype('object').fillna('Unknown')
    return X

def run_cv_catboost(X, y, X_test=None, n_splits=5, seed=SEED, early_stopping=400):
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)

    # 先做特征工程，再识别类别列，再做缺失填充
    X_fe = base_feature_engineering(X)
    X_test_fe = base_feature_engineering(X_test, is_train=False) if X_test is not None else None

    cat_cols = get_cat_cols(X_fe)
    X_fe = fill_missing(X_fe, cat_cols)
    if X_test_fe is not None:
        X_test_fe = fill_missing(X_test_fe, cat_cols)

    # 统一类型，避免 Pandas 扩展 Int64 导致的 CatBoost Pool 错误
    X_fe = ensure_catboost_dtypes(X_fe, cat_cols)
    if X_test_fe is not None:
        X_test_fe = ensure_catboost_dtypes(X_test_fe, cat_cols)

    # 特征列（去掉 id/label 等非特征列）
    feat_cols = [c for c in X_fe.columns if c not in NON_FEATURE_COLS]

    def to_pool(df, y=None):
        # 计算在 feat_cols 上的类别列索引
        cat_idx = [feat_cols.index(c) for c in cat_cols if c in feat_cols]
        return Pool(df[feat_cols], label=y, cat_features=cat_idx)

    oof = np.zeros(len(X_fe))
    test_pred = np.zeros(len(X_test_fe)) if X_test_fe is not None else None
    best_iterations, fold_auc = [], []

    default_params = dict(
        loss_function='Logloss', eval_metric='AUC', iterations=8000, learning_rate=0.02,
        depth=6, l2_leaf_reg=10.0, random_seed=seed, bootstrap_type='Bayesian',
        bagging_temperature=0.5, rsm=0.8, random_strength=2.0, border_count=128,
        grow_policy='SymmetricTree', od_type='Iter', od_wait=early_stopping,
        verbose=200, allow_writing_files=False, auto_class_weights='Balanced'
    )

    for fold, (tr_idx, va_idx) in enumerate(skf.split(X_fe, y), 1):
        X_tr, X_va = X_fe.iloc[tr_idx], X_fe.iloc[va_idx]
        y_tr, y_va = y.iloc[tr_idx], y.iloc[va_idx]

        model = CatBoostClassifier(**default_params)
        model.fit(to_pool(X_tr, y_tr), eval_set=to_pool(X_va, y_va), verbose=200)
        va_pred = model.predict_proba(to_pool(X_va))[:, 1] if hasattr(model, 'predict_proba') else model.predict(to_pool(X_va))
        if hasattr(va_pred, 'ravel'): va_pred = va_pred.ravel()
        oof[va_idx] = va_pred
        va_auc = roc_auc_score(y_va, va_pred); fold_auc.append(va_auc)
        best_iterations.append(model.get_best_iteration())
        if X_test_fe is not None:
            test_pred += model.predict_proba(to_pool(X_test_fe))[:, 1] / n_splits
        print(f"[Fold {fold}] AUC = {va_auc:.6f} | best_iter = {model.get_best_iteration()}")

    cv_auc = roc_auc_score(y, oof)
    print(f"OOF AUC = {cv_auc:.6f}; folds = {[round(a,6) for a in fold_auc]}")
    return dict(oof=oof, test_pred=test_pred, cv_auc=cv_auc, fold_auc=fold_auc,
                best_iterations=best_iterations, cat_cols=cat_cols,
                X=X_fe[feat_cols], X_test=(X_test_fe[feat_cols] if X_test_fe is not None else None))

def retrain_full_and_predict(X, y, X_test, cat_cols, best_iterations, seed=SEED):
    if X_test is None: return None
    best_iter=int(np.mean(best_iterations))
    params=dict(loss_function='Logloss', eval_metric='AUC', iterations=max(200,best_iter), learning_rate=0.02,
                depth=6, l2_leaf_reg=10.0, random_seed=seed, bootstrap_type='Bayesian', bagging_temperature=0.5,
                rsm=0.8, random_strength=2.0, border_count=128, grow_policy='SymmetricTree', verbose=200,
                allow_writing_files=False, auto_class_weights='Balanced')
    cat_idx=[X.columns.get_loc(c) for c in cat_cols if c in X.columns]
    model=CatBoostClassifier(**params)
    model.fit(Pool(X, label=y, cat_features=cat_idx), verbose=200)
    preds=model.predict_proba(X_test)[:,1]
    return preds

def ensure_catboost_dtypes(df: pd.DataFrame, cat_cols):
    """将 DataFrame 转成 CatBoost 友好类型：
    - 类别列 -> str
    - 其他特征 -> float32
    - 丢弃不可用的扩展整数(Int64等)
    """
    X = df.copy()
    # 类别列先统一为字符串
    for c in cat_cols:
        if c in X.columns:
            X[c] = X[c].astype(str)
    # 其他特征转换为 float32
    for c in X.columns:
        if c in cat_cols or c in NON_FEATURE_COLS:
            continue
        X[c] = pd.to_numeric(X[c], errors='coerce').astype('float32')
    return X


In [8]:
X_raw,y,X_test_raw=prepare_dataset(TRAIN_CSV, TRAIN_STMT_FEAT, TEST_CSV, TEST_STMT_FEAT)
all_test_pred=None
info_all=[]
for s in ENSEMBLE_SEEDS:
    np.random.seed(s)
    cv=run_cv_catboost(X_raw,y,X_test=X_test_raw,seed=s)
    oof_df=pd.DataFrame({'id': X_raw['id'].values, 'oof_pred': cv['oof'], 'label': y.values})
    oof_df.to_csv(SAVE_OOF.replace('.csv', f'_seed{s}.csv'), index=False, encoding='utf-8')
    info_all.append({'seed': s, 'cv_auc': float(cv['cv_auc']), 'fold_auc': [float(x) for x in cv['fold_auc']], 'best_iterations': [int(x) for x in cv['best_iterations']]})
    if X_test_raw is not None:
        preds=retrain_full_and_predict(cv['X'], y, cv['X_test'], cv['cat_cols'], cv['best_iterations'], seed=s)
        all_test_pred = preds if all_test_pred is None else (all_test_pred + preds)
if X_test_raw is not None and all_test_pred is not None:
    all_test_pred/=len(ENSEMBLE_SEEDS)
    sub=pd.DataFrame({'id': X_test_raw['id'].values, 'prob': all_test_pred})
    sub.to_csv(OUT_SUBMISSION, index=False, encoding='utf-8')
    print('提交文件保存：', OUT_SUBMISSION, sub.shape)
with open(SAVE_INFO, 'w', encoding='utf-8') as f:
    json.dump({'seeds': info_all}, f, ensure_ascii=False, indent=2)
print('CV 信息保存：', SAVE_INFO)

0:	test: 0.6105463	best: 0.6105463 (0)	total: 134ms	remaining: 17m 51s
200:	test: 0.6444860	best: 0.6445296 (199)	total: 11.1s	remaining: 7m 11s
400:	test: 0.6461936	best: 0.6465740 (372)	total: 23.2s	remaining: 7m 19s
600:	test: 0.6476586	best: 0.6477024 (599)	total: 32.8s	remaining: 6m 44s
800:	test: 0.6477859	best: 0.6482746 (667)	total: 43.3s	remaining: 6m 29s
1000:	test: 0.6470662	best: 0.6482746 (667)	total: 54.3s	remaining: 6m 19s
Stopped by overfitting detector  (400 iterations wait)

bestTest = 0.648274575
bestIteration = 667

Shrink model to first 668 iterations.
[Fold 1] AUC = 0.648275 | best_iter = 667
0:	test: 0.6104027	best: 0.6104027 (0)	total: 59.1ms	remaining: 7m 52s
200:	test: 0.6360629	best: 0.6373956 (12)	total: 9s	remaining: 5m 49s
400:	test: 0.6398022	best: 0.6398022 (400)	total: 20.9s	remaining: 6m 36s
600:	test: 0.6432535	best: 0.6433547 (590)	total: 30.8s	remaining: 6m 18s
800:	test: 0.6460248	best: 0.6460248 (800)	total: 41.5s	remaining: 6m 13s
1000:	test: 0.6